In [10]:
import pandas as pd
import numpy as np
# Load the input data
input_data = pd.read_excel('DATest_1.xlsx', sheet_name='Drivers_Input')
input_data

,User Id,Keyword,Vibe,Keyword Notes
0,0030b069-d45e-11ee-8482-128e718ba88f,Drivers,1,NaN
1,00831062-d45b-11ee-8482-128e718ba88f,Drivers,1,NaN
2,00ac49e8-d45a-11ee-8482-128e718ba88f,Drivers,3,NaN
3,00fd1031-d466-11ee-8482-128e718ba88f,Drivers,3,NaN
4,011c0f25-d464-11ee-8482-128e718ba88f,Drivers,1,NaN
...,...,...,...,...
5248,fc3b09c3-d465-11ee-8482-128e718ba88f,Drivers,1,Tough
5249,fc3b09c3-d465-11ee-8482-128e718ba88f,Drivers,1,Unmissable Presence
5250,fd339b8a-d469-11ee-8482-128e718ba88f,Drivers,1,Convenience
5251,fd339b8a-d469-11ee-8482-128e718ba88f,Drivers,3,Dealership Sales Experience


In [4]:
# Create a pivot table that counts occurrences of each Keyword with Vibe values
pivot_table = pd.pivot_table(input_data, 
                            index='Keyword', 
                            columns='Vibe', 
                            aggfunc='size', 
                            fill_value=0)


In [21]:
# Get unique keyword notes for the "Overall: Drivers" row
unique_notes = input_data['Keyword Notes'].dropna().unique()
overall_drivers_value = ', '.join(unique_notes) if len(unique_notes) > 0 else ''

# Define the mapping between Vibe numbers and categories
# Note: In the desired output, 1=Positive, 2=Negative, 3=Mixed, blank=Neutral
vibe_mapping = {1: 'Positive', 2: 'Negative', 3: 'Mixed', np.nan: 'Neutral'}

# Create pivot table counting occurrences of each Keyword with Vibe values
pivot_table = pd.pivot_table(input_data,
                            index='Keyword',
                            columns='Vibe',
                            aggfunc='size',
                            fill_value=0)

# Ensure all columns are present (add missing ones with zeros)
for col in vibe_mapping.keys():
    if col not in pivot_table.columns:
        pivot_table[col] = 0

# Reorder and rename columns to match desired output
pivot_table = pivot_table[[1, np.nan, 3, 2]]  # Positive, Neutral, Mixed, Negative
pivot_table.columns = ['Positive', 'Neutral', 'Mixed', 'Negative']

# Calculate totals
pivot_table['Grand Total'] = pivot_table.sum(axis=1)

# Calculate percentages (rounded to whole numbers)
percentages = (pivot_table[['Positive', 'Neutral', 'Mixed', 'Negative']]
              .div(pivot_table['Grand Total'], axis=0) * 100)
percentages = percentages.round(0).astype(int)

# Calculate combined percentage (Positive + 0.5*Neutral)
combined_pct = ((pivot_table['Positive'] + pivot_table['Neutral'] * 0.5) / 
               pivot_table['Grand Total'] * 100).round(0).astype(int)

# Build final output dataframe
final_output = pd.DataFrame({
    '(blank)': '',
    '1': pivot_table['Positive'],  # Positive count
    '(blank)': '',
    '3': pivot_table['Mixed'],     # Mixed count
    '2': pivot_table['Negative'],  # Negative count
    'Grand Total': pivot_table['Grand Total'],
    'Positive': percentages['Positive'].astype(str) + '%',
    'Neutral': percentages['Neutral'].astype(str) + '%',
    'Mixed': percentages['Mixed'].astype(str) + '%',
    'Negative': percentages['Negative'].astype(str) + '%',
    'Grand Total%': combined_pct.astype(str) + '%'
})

# Sort by Grand Total in descending order
final_output = final_output.sort_values('Grand Total', ascending=False)

# Create the total row (first row in the desired output)
total_row = pd.DataFrame({
    '(blank)': ['Overall: Drivers'],
    '1': [overall_drivers_value],  # Insert unique keyword notes here
    '(blank)': [''],
    '3': [''],
    '2': [''],
    'Grand Total': [''],
    'Positive': [''],
    'Neutral': [''],
    'Mixed': [''],
    'Negative': [''],
    'Grand Total%': ['']
}, index=[0])

# Create the blank row (second row in the desired output)
blank_row = pd.DataFrame({
    '(blank)': ['(blank)'],
    '1': [pivot_table['Positive'].sum()],
    '(blank)': [''],
    '3': [pivot_table['Mixed'].sum()],
    '2': [pivot_table['Negative'].sum()],
    'Grand Total': [pivot_table['Grand Total'].sum()],
    'Positive': [''],
    'Neutral': [''],
    'Mixed': [''],
    'Negative': [''],
    'Grand Total%': ['']
}, index=[1])

# Combine all rows
final_output = pd.concat([total_row, blank_row, final_output])

# Reset index
final_output.reset_index(drop=True, inplace=True)
final_output
# Save to CSV
final_output.to_csv('Drivers_DerisedOutput.csv', index=False)